# Building a Coding Agent with OpenVINO

**Notebook 1 of 3** — Agent Construction

This notebook walks through building a full coding agent using:
- **Model**: `Qwen3-Coder-30B-A3B-Instruct` or `gpt-oss-20b`
- **Server**: OpenVINO Model Server (OVMS) with tool parsing
- **Agent loop**: OpenAI-compatible client with tool calling

**Prerequisites**: Ubuntu 24.04 Noble, Intel drivers installed (see README)

---

## 1. Environment Setup

In [2]:
# Install required packages
import sys
!{sys.executable} -m pip install -q \
    openvino \
    openvino-genai \
    openvino-tokenizers \
    optimum-intel \
    nncf \
    openai \
    huggingface_hub \
    ipywidgets \
    rich

In [5]:
import openvino as ov
import subprocess
import sys
import os
import json
import time
import textwrap
from pathlib import Path
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from rich.table import Table

console = Console()

# Detect available devices
core = ov.Core()
available = core.available_devices

table = Table(title="Available OpenVINO Devices")
table.add_column("Device", style="cyan")
table.add_column("Full Name", style="green")
for d in available:
    try:
        name = core.get_property(d, "FULL_DEVICE_NAME")
    except:
        name = "N/A"
    table.add_row(d, name)

console.print(table)

           Available OpenVINO Devices           
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Device ┃ Full Name                           ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ CPU    │ 12th Gen Intel(R) Core(TM) i7-1255U │
└────────┴─────────────────────────────────────┘

## 2. Model Selection & Export

Choose your model and target device below. The export converts the HuggingFace model to OpenVINO IR format with INT4 quantization.

In [12]:
import ipywidgets as widgets
from IPython.display import display

model_dropdown = widgets.Dropdown(
    options=[
        ("Qwen3-Coder-30B-A3B-Instruct (best coding, large)", "Qwen/Qwen3-Coder-30B-A3B-Instruct"),
        ("Qwen3-8B (faster, smaller)", "Qwen/Qwen3-8B"),
        ("gpt-oss-20b (OpenAI open model)", "openai/gpt-oss-20b"),
    ],
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px")
)

device_dropdown = widgets.Dropdown(
    options=["GPU", "CPU", "NPU", "AUTO"],
    value="GPU",
    description="Target Device:",
    style={"description_width": "initial"}
)

display(model_dropdown, device_dropdown)

Dropdown(description='Model:', layout=Layout(width='520px'), options=(('Qwen3-Coder-30B-A3B-Instruct (best cod…

Dropdown(description='Target Device:', options=('GPU', 'CPU', 'NPU', 'AUTO'), style=DescriptionStyle(descripti…

In [13]:
MODEL_ID = model_dropdown.value
TARGET_DEVICE = device_dropdown.value
MODEL_NAME = MODEL_ID.split("/")[-1]
MODEL_PATH = Path(f"{MODEL_NAME}-int4-ov")

# Tool parser mapping per model
TOOL_PARSER_MAP = {
    "Qwen3-Coder-30B-A3B-Instruct": "qwen3coder",
    "Qwen3-8B":                     "hermes3",
    "gpt-oss-20b":                  "gptoss",
}
REASONING_PARSER_MAP = {
    "Qwen3-8B":    "qwen3",
    "gpt-oss-20b": "gptoss",
}

TOOL_PARSER     = TOOL_PARSER_MAP.get(MODEL_NAME, "hermes3")
REASONING_PARSER = REASONING_PARSER_MAP.get(MODEL_NAME, "")

console.print(Panel(
    f"[bold]Model:[/bold]     {MODEL_ID}\n"
    f"[bold]Device:[/bold]    {TARGET_DEVICE}\n"
    f"[bold]Export to:[/bold] {MODEL_PATH}\n"
    f"[bold]Tool Parser:[/bold] {TOOL_PARSER}",
    title="Configuration", border_style="cyan"
))

╭───────────────────────────────────────────────── Configuration ─────────────────────────────────────────────────╮
│ Model:     Qwen/Qwen3-Coder-30B-A3B-Instruct                                                                    │
│ Device:    GPU                                                                                                  │
│ Export to: Qwen3-Coder-30B-A3B-Instruct-int4-ov                                                                 │
│ Tool Parser: qwen3coder                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Export model to OpenVINO IR format (skip if already exported)
if MODEL_PATH.exists():
    console.print(f"[green]✓ Model already exported at {MODEL_PATH}[/green]")
else:
    console.print(f"[yellow]Exporting {MODEL_ID} → {MODEL_PATH} ...[/yellow]")
    console.print("[dim]This may take 10–30 min depending on model size and hardware[/dim]")

    # gpt-oss requires --pipeline_type LM
    extra = "--pipeline_type LM" if "gpt-oss" in MODEL_NAME else ""

    # NPU requires specific quantization flags
    if TARGET_DEVICE == "NPU":
        quant_args = "--weight-format int4 --sym --ratio 1.0 --group-size -1"
    else:
        quant_args = "--weight-format int4 --scale-estimation --dataset wikitext2"

    cmd = (
        f"optimum-cli export openvino "
        f"--model {MODEL_ID} "
        f"--task text-generation-with-past "
        f"{quant_args} "
        f"{extra} "
        f"{MODEL_PATH}"
    )
    console.print(Syntax(cmd, "bash", theme="monokai"))
    ret = subprocess.run(cmd, shell=True)
    if ret.returncode == 0:
        console.print("[green]✓ Export complete[/green]")
    else:
        console.print("[red]✗ Export failed — check logs above[/red]")

Exporting Qwen/Qwen3-Coder-30B-A3B-Instruct → Qwen3-Coder-30B-A3B-Instruct-int4-ov ...

This may take 10–30 min depending on model size and hardware

optimum-cli export openvino --model Qwen/Qwen3-Coder-30B-A3B-Instruct --task text-generation-with-past --weight-for

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

## 3. Start OpenVINO Model Server (OVMS)

OVMS serves the model via an OpenAI-compatible REST API with built-in tool parsing.

In [ ]:
# Check if Docker is available, otherwise use bare-metal OVMS binary
def check_docker():
    r = subprocess.run(["docker", "info"], capture_output=True)
    return r.returncode == 0

def check_ovms_binary():
    r = subprocess.run(["ovms", "--version"], capture_output=True)
    return r.returncode == 0

USE_DOCKER = check_docker()
USE_BINARY = check_ovms_binary()

console.print(f"Docker available:      {'[green]Yes[/green]' if USE_DOCKER else '[red]No[/red]'}")
console.print(f"OVMS binary available: {'[green]Yes[/green]' if USE_BINARY else '[red]No[/red]'}")

if not USE_DOCKER and not USE_BINARY:
    console.print(Panel(
        "Install OVMS:\n"
        "  Docker: https://docs.docker.com/engine/install/ubuntu/\n"
        "  Binary: https://docs.openvino.ai/2025/model-server/ovms_what_is_openvino_model_server.html",
        title="[red]OVMS not found[/red]", border_style="red"
    ))

In [ ]:
import subprocess

OVMS_PORT = 8000
MODELS_DIR = Path("./models").resolve()
MODELS_DIR.mkdir(exist_ok=True)

reasoning_flag = f"--reasoning_parser {REASONING_PARSER}" if REASONING_PARSER else ""
pipeline_flag  = "--pipeline_type LM" if "gpt-oss" in MODEL_NAME else ""

if USE_DOCKER:
    # GPU-enabled Docker image
    gpu_flags = ""
    if TARGET_DEVICE in ("GPU", "NPU"):
        render_gid = subprocess.check_output(
            "stat -c '%g' /dev/dri/render* | head -n 1", shell=True
        ).decode().strip()
        gpu_flags = f"--device /dev/dri --group-add={render_gid}"
        if TARGET_DEVICE == "NPU":
            gpu_flags += " --device /dev/accel"

    docker_cmd = f"""\
docker run -d --name ovms_coding_agent \\
  --user $(id -u):$(id -g) --rm \\
  -p {OVMS_PORT}:{OVMS_PORT} \\
  -v {MODELS_DIR}:/models \\
  {gpu_flags} \\
  openvino/model_server:latest-gpu \\
  --rest_port {OVMS_PORT} \\
  --source_model {MODEL_ID} \\
  --model_repository_path /models \\
  --tool_parser {TOOL_PARSER} \\
  {reasoning_flag} \\
  --target_device {TARGET_DEVICE} \\
  --task text_generation \\
  --enable_tool_guided_generation true \\
  --enable_prefix_caching true \\
  {pipeline_flag}"""

    console.print(Panel(Syntax(docker_cmd, "bash", theme="monokai"), title="Docker Launch Command"))
    console.print("[yellow]Run the command above in a terminal, then continue.[/yellow]")

elif USE_BINARY:
    binary_cmd = f"""\
ovms \\
  --rest_port {OVMS_PORT} \\
  --source_model {MODEL_ID} \\
  --model_repository_path {MODELS_DIR} \\
  --tool_parser {TOOL_PARSER} \\
  {reasoning_flag} \\
  --target_device {TARGET_DEVICE} \\
  --task text_generation \\
  --enable_tool_guided_generation true \\
  --enable_prefix_caching true \\
  {pipeline_flag}"""

    console.print(Panel(Syntax(binary_cmd, "bash", theme="monokai"), title="Binary Launch Command"))
    console.print("[yellow]Run the command above in a separate terminal, then continue.[/yellow]")

In [ ]:
# Wait for OVMS to be ready
import urllib.request

def wait_for_ovms(port=8000, timeout=120):
    url = f"http://localhost:{port}/v3/models"
    console.print(f"[yellow]Waiting for OVMS on port {port}...[/yellow]")
    start = time.time()
    while time.time() - start < timeout:
        try:
            urllib.request.urlopen(url, timeout=2)
            console.print("[green]✓ OVMS is ready![/green]")
            return True
        except Exception:
            time.sleep(3)
            print(".", end="", flush=True)
    console.print("\n[red]✗ OVMS did not start in time[/red]")
    return False

ovms_ready = wait_for_ovms()

## 4. Define Coding Agent Tools

The agent can execute code, read/write files, run shell commands, and search for patterns.

In [ ]:
import subprocess
import tempfile

# --- Tool definitions (sent to the model) ---
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Execute Python code in an isolated subprocess and return stdout/stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string", "description": "Python code to execute"},
                    "timeout": {"type": "integer", "description": "Max seconds to run (default 15)", "default": 15}
                },
                "required": ["code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_shell",
            "description": "Execute a shell command and return output.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "Shell command to run"},
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write content to a file on disk.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path to write"},
                    "content": {"type": "string", "description": "File content"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read content from a file on disk.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path to read"}
                },
                "required": ["path"]
            }
        }
    },
]

# --- Tool implementations ---
def run_python(code: str, timeout: int = 15) -> str:
    with tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False) as f:
        f.write(code)
        fname = f.name
    try:
        result = subprocess.run(
            [sys.executable, fname],
            capture_output=True, text=True, timeout=timeout
        )
        out = result.stdout or ""
        err = result.stderr or ""
        return (out + ("\nSTDERR:\n" + err if err else "")).strip()
    except subprocess.TimeoutExpired:
        return f"[TIMEOUT after {timeout}s]"
    finally:
        os.unlink(fname)

def run_shell(command: str) -> str:
    result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
    return (result.stdout + result.stderr).strip()

def write_file(path: str, content: str) -> str:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(content)
    return f"Written {len(content)} chars to {path}"

def read_file(path: str) -> str:
    return Path(path).read_text()

TOOL_DISPATCH = {
    "run_python": run_python,
    "run_shell":  run_shell,
    "write_file": write_file,
    "read_file":  read_file,
}

console.print("[green]✓ Tools defined:[/green]", [t["function"]["name"] for t in TOOLS])

## 5. Agent Loop

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f"http://localhost:{OVMS_PORT}/v3",
    api_key="unused"
)

SYSTEM_PROMPT = """\
You are an expert coding agent. When given a programming task:
1. Plan your approach briefly
2. Write clean, well-commented code
3. Use run_python to test it immediately
4. Fix any errors and re-test
5. Return the final working solution

Always verify your code actually runs before presenting it as a solution.
"""

def run_agent(
    user_request: str,
    max_turns: int = 10,
    verbose: bool = True
) -> dict:
    """Run the coding agent on a task. Returns result dict with timing."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_request},
    ]

    total_start = time.perf_counter()
    turn_count = 0
    tool_calls_made = []

    while turn_count < max_turns:
        turn_count += 1
        if verbose:
            console.print(f"\n[bold cyan]--- Turn {turn_count} ---[/bold cyan]")

        t0 = time.perf_counter()
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            tools=TOOLS,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        latency = time.perf_counter() - t0

        msg = response.choices[0].message
        usage = response.usage

        if verbose and msg.content:
            console.print(Panel(msg.content[:600] + ("..." if len(msg.content) > 600 else ""),
                                title=f"Agent (turn {turn_count})", border_style="blue"))

        # Convert message to dict for history
        msg_dict = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            msg_dict["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(msg_dict)

        # No tool calls = agent is done
        if not msg.tool_calls:
            break

        # Execute each tool call
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            tool_calls_made.append(name)

            if verbose:
                console.print(f"  [yellow]→ Tool:[/yellow] {name}({list(args.keys())}")

            fn = TOOL_DISPATCH.get(name)
            if fn:
                result = fn(**args)
            else:
                result = f"Unknown tool: {name}"

            if verbose:
                preview = str(result)[:200]
                console.print(f"  [green]← Result:[/green] {preview}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result)
            })

    total_time = time.perf_counter() - total_start
    final_answer = next(
        (m["content"] for m in reversed(messages)
         if m["role"] == "assistant" and m["content"]),
        "No answer"
    )

    return {
        "answer": final_answer,
        "turns": turn_count,
        "tool_calls": tool_calls_made,
        "total_time_s": round(total_time, 2),
        "messages": messages,
    }

console.print("[green]✓ Agent loop ready[/green]")

## 6. Run Example Tasks

In [ ]:
# Task 1: Algorithm implementation
if ovms_ready:
    result = run_agent(
        "Implement a binary search tree in Python with insert, search, "
        "and inorder traversal. Write and run tests to verify it works correctly."
    )
    console.print(f"\n[bold]Completed in {result['total_time_s']}s | "
                  f"{result['turns']} turns | tools: {result['tool_calls']}[/bold]")
else:
    console.print("[red]OVMS not running — start it first[/red]")

In [ ]:
# Task 2: Debugging task
if ovms_ready:
    buggy_code = '''
def find_duplicates(lst):
    seen = []
    duplicates = []
    for item in lst:
        if item in seen:
            seen.append(item)  # bug: should append to duplicates
        else:
            seen.append(item)
    return duplicates

print(find_duplicates([1, 2, 3, 2, 4, 3, 5]))  # expected: [2, 3]
'''
    result2 = run_agent(
        f"Debug and fix this Python function:\n```python\n{buggy_code}\n```"
    )
    console.print(f"\n[bold]Completed in {result2['total_time_s']}s[/bold]")

## 7. Save Agent for Use in Benchmark Notebooks

In [ ]:
# Save config so benchmark notebooks can import it
config = {
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "model_path": str(MODEL_PATH),
    "tool_parser": TOOL_PARSER,
    "reasoning_parser": REASONING_PARSER,
    "ovms_port": OVMS_PORT,
}
Path("agent_config.json").write_text(json.dumps(config, indent=2))
console.print("[green]✓ agent_config.json saved — use this in benchmark notebooks[/green]")
console.print_json(json.dumps(config, indent=2))

---
## ✅ Summary

You've built a working coding agent with:
- OpenVINO-accelerated inference via OVMS
- Tool calling: `run_python`, `run_shell`, `write_file`, `read_file`
- Multi-turn agent loop with structured output

**Next steps:**
- `02_benchmark_pantherlake.ipynb` — benchmark on Core Ultra Series 3 (Xe3 Arc B390)
- `03_benchmark_lunarlake.ipynb` — benchmark on Core Ultra Series 2 (Xe2 Arc 140V)